In [1]:
import ufl
from dolfinx import fem
from mpi4py import MPI
from dolfinx.io import gmsh
import vtk
import meshio
import numpy as np
from vtk.util.numpy_support import vtk_to_numpy
from scipy.spatial import cKDTree
from dolfinx import mesh, fem, io, plot
from scipy.interpolate import RBFInterpolator
from types import SimpleNamespace
import basix
import dolfinx.fem.petsc
from dolfinx.fem.petsc import NonlinearProblem
from pathlib import Path



# LOCAL FRAME
def normalize(v):
    return v / ufl.sqrt(ufl.dot(v, v))

def _local_frame_ufl(domain):
    t  = ufl.Jacobian(domain)
    t1 = ufl.as_vector([t[0, 0], t[1, 0], t[2, 0]])
    t2 = ufl.as_vector([t[0, 1], t[1, 1], t[2, 1]])
    e3 = normalize(ufl.cross(t1, t2))
    ey = ufl.as_vector([0, 1, 0])
    ez = ufl.as_vector([0, 0, 1])
    e1_trial = ufl.cross(ey, e3)
    norm_e1  = ufl.sqrt(ufl.dot(e1_trial, e1_trial))
    e1 = ufl.conditional(ufl.lt(norm_e1, 0.5), ez, normalize(e1_trial))
    e2 = normalize(ufl.cross(e3, e1))
    return e1, e2, e3

def local_frame(domain, gdim):
    FRAME  = _local_frame_ufl(domain)
    VT     = fem.functionspace(domain, ("DG", 0, (gdim,)))
    V0, _  = VT.sub(0).collapse()
    BASIS  = [fem.Function(VT, name=f"Basis_vector_e{i+1}") for i in range(gdim)]
    for i in range(gdim):
        e_exp = fem.Expression(FRAME[i], V0.element.interpolation_points)
        BASIS[i].interpolate(e_exp)
    return BASIS[0], BASIS[1], BASIS[2]


# SHELL KINEMATICS
def hstack(vecs):
    return ufl.as_matrix([[vi[i] for i in range(len(vi))] for vi in vecs]).T

def tangent_projection(e1, e2):
    return hstack([e1, e2])

def tangential_gradient(w, P):
    return ufl.dot(ufl.grad(w), P)

def membrane_strain(u, P):
    t_gu = ufl.dot(P.T, tangential_gradient(u, P))
    return ufl.sym(t_gu), t_gu

def bending_strain(theta, e3, P):
    beta = ufl.cross(e3, theta)
    return ufl.sym(ufl.dot(P.T, tangential_gradient(beta, P)))

def shear_strain(u, theta, e3, P):
    beta = ufl.cross(e3, theta)
    return tangential_gradient(ufl.dot(u, e3), P) - ufl.dot(P.T, beta)

def compute_drilling_strain(t_gu, theta, e3):
    return (t_gu[0, 1] - t_gu[1, 0]) / 2 + ufl.dot(theta, e3)

def shell_strains(u, theta, e1, e2, e3):
    P                 = tangent_projection(e1, e2)
    eps, t_gu         = membrane_strain(u, P)
    kappa             = bending_strain(theta, e3, P)
    gamma             = shear_strain(u, theta, e3, P)
    drilling_strain   = compute_drilling_strain(t_gu, theta, e3)
    return eps, kappa, gamma, drilling_strain



# ISOTROPIC MATERIAL
def isotropic_material(thickness, young, poisson, domain):
    h        = fem.Constant(domain, float(thickness))
    E        = fem.Constant(domain, float(young))
    nu       = fem.Constant(domain, float(poisson))
    lmbda    = E * nu / (1 + nu) / (1 - 2 * nu)
    mu       = E / 2 / (1 + nu)
    lmbda_ps = 2 * lmbda * mu / (lmbda + 2 * mu)
    return SimpleNamespace(h=h, E=E, nu=nu, lmbda=lmbda, mu=mu, lmbda_ps=lmbda_ps, kind="isotropic")

# CLT COMPOSITE MATERIAL
def _Q_ply(E1, E2, G12, nu12):
    nu21 = nu12 * E2 / E1
    d    = 1 - nu12 * nu21
    return np.array([
        [ E1 / d,        nu12 * E2 / d,  0   ],
        [ nu12 * E2 / d, E2 / d,         0   ],
        [ 0,             0,              G12 ],
    ])


def _Qbar_ply(Q, angle_deg):
    a    = np.radians(angle_deg)
    c, s = np.cos(a), np.sin(a)
    T = np.array([
        [ c**2,   s**2,   2*c*s          ],
        [ s**2,   c**2,  -2*c*s          ],
        [-c*s,    c*s,    c**2 - s**2    ],
    ])
    R    = np.diag([1.0, 1.0, 2.0])
    Rinv = np.diag([1.0, 1.0, 0.5])
    return np.linalg.inv(T) @ Q @ R @ T @ Rinv

def compute_ABD(layup_angles, t_ply, E1, E2, G12, nu12):
    Q   = _Q_ply(E1, E2, G12, nu12)
    H   = t_ply * len(layup_angles)
    z   = -H / 2.0
    A   = np.zeros((3, 3))
    B   = np.zeros((3, 3))
    D   = np.zeros((3, 3))
    for angle in layup_angles:
        Qb = _Qbar_ply(Q, angle)
        z0, z1 = z, z + t_ply
        A += Qb * (z1 - z0)
        B += Qb * (z1**2 - z0**2) / 2.0
        D += Qb * (z1**3 - z0**3) / 3.0
        z  = z1
    return A, B, D, H

def clt_material(layup_angles, t_ply,E1, E2, G12, nu12,G13=None, G23=None, kappa_s=5/6):
    if G13 is None: G13 = G12
    if G23 is None: G23 = G12 * 0.5

    A_np, B_np, D_np, H = compute_ABD(layup_angles, t_ply, E1, E2, G12, nu12)

    max_B = np.abs(B_np).max()
    print(f"[CLT] Layup  : {layup_angles}")
    print(f"[CLT] H      : {H*1e3:.2f} mm")
    print(f"[CLT] max|B| : {max_B:.2e}  {'SYMMETRIC' if max_B < 1e-6 * A_np.max() else 'NON-SYMMETRIC'}")
    print(f"[CLT] A11    : {A_np[0,0]/1e6:.2f} MPa·m")
    print(f"[CLT] D11    : {D_np[0,0]:.4f} N·m²")

    As_np = kappa_s * H * np.array([
        [ G13, 0.0 ],
        [ 0.0, G23 ],
    ])

    # EFFECTIVE IN-PLANE SHEAR FOR DRILLING STABILISATION
    G_eff = float(A_np[2, 2]) / H

    return SimpleNamespace(
        kind   = "clt",
        H      = H,
        A_np   = A_np,
        B_np   = B_np,
        D_np   = D_np,
        As_np  = As_np,
        G_eff  = G_eff,
        A_ufl  = ufl.as_tensor(A_np),
        B_ufl  = ufl.as_tensor(B_np),
        D_ufl  = ufl.as_tensor(D_np),
        As_ufl = ufl.as_tensor(As_np),
    )

# VOIGT NOTATION
def to_voigt(e):
    return ufl.as_vector([e[0, 0], e[1, 1], 2.0 * e[0, 1]])


def from_voigt(v):
    return ufl.as_tensor([
        [v[0], v[2]],
        [v[2], v[1]],
    ])


# SHELL STRESS RESULTANTS
def _plane_stress_iso(mat, e):
    tdim = e.ufl_shape[0]
    return mat.lmbda_ps * ufl.tr(e) * ufl.Identity(tdim) + 2 * mat.mu * e


def stress_resultants(mat, eps, kappa, gamma):
    if mat.kind == "isotropic":
        N = mat.h * _plane_stress_iso(mat, eps)
        M = mat.h**3 / 12.0 * _plane_stress_iso(mat, kappa)
        Q = mat.mu * mat.h * gamma

    elif mat.kind == "clt":
        eps_v   = to_voigt(eps)
        kappa_v = to_voigt(kappa)

        N_v = ufl.dot(mat.A_ufl, eps_v) + ufl.dot(mat.B_ufl, kappa_v)
        M_v = ufl.dot(mat.B_ufl, eps_v) + ufl.dot(mat.D_ufl, kappa_v)

        N   = from_voigt(N_v)
        M   = from_voigt(M_v)
        Q   = ufl.dot(mat.As_ufl, gamma)

    else:
        raise ValueError(f"Unknown material kind: {mat.kind!r}")
    return N, M, Q


def drilling_terms(mat, domain, drilling_strain):
    h_mesh = ufl.CellDiameter(domain)

    if mat.kind == "isotropic":
        G_eff = mat.mu
    else:
        G_eff = fem.Constant(domain, mat.G_eff)

    h = mat.h if mat.kind == "isotropic" else fem.Constant(domain, mat.H)

    stiffness = G_eff * h**3 / h_mesh**2
    stress    = stiffness * drilling_strain
    return stiffness, stress


# IMPORT AND MAP OPENFOAM DATA
def import_foam_traction(foamfile, xdmffile, verbose=False):
    reader = vtk.vtkXMLPolyDataReader()
    reader.SetFileName(foamfile)
    reader.Update()
    poly = reader.GetOutput()

    tri = vtk.vtkTriangleFilter()
    tri.SetInputData(poly)
    tri.Update()
    poly = tri.GetOutput()

    points = vtk_to_numpy(poly.GetPoints().GetData())
    p      = vtk_to_numpy(poly.GetPointData().GetArray("p"))
    wss    = vtk_to_numpy(poly.GetPointData().GetArray("wallShearStress"))

    cells = poly.GetPolys()
    cells.InitTraversal()
    idList = vtk.vtkIdList()
    triangles = []
    while cells.GetNextCell(idList):
        triangles.append([idList.GetId(i) for i in range(3)])
    triangles = np.array(triangles)

    P0 = points[triangles[:, 0]]
    P1 = points[triangles[:, 1]]
    P2 = points[triangles[:, 2]]

    v1 = P1 - P0
    v2 = P2 - P0
    n  = np.cross(v1, v2)

    area = 0.5 * np.linalg.norm(n, axis=1)
    n_unit = n / np.linalg.norm(n, axis=1)[:, None]

    p_tri   = p[triangles].mean(axis=1)
    wss_tri = wss[triangles].mean(axis=1)

    traction_cell = -p_tri[:, None] * n_unit + wss_tri

    nodal_traction = np.zeros_like(points)
    nodal_area     = np.zeros(len(points))

    for i, tri in enumerate(triangles):
        for j in tri:
            nodal_traction[j] += traction_cell[i] * area[i] / 3.0
            nodal_area[j]     += area[i] / 3.0

    nodal_traction /= nodal_area[:, None]

    if verbose:
        total_force = (traction_cell * area[:, None]).sum(axis=0)
        print("[FOAM] Total OpenFOAM force:", total_force)

    meshio.write(xdmffile, meshio.Mesh(
        points=points,
        cells=[("triangle", triangles)],
        point_data={"traction": nodal_traction},
    ))

    print(f"[FOAM] Exported traction to the file {xdmffile}")

def map_traction(foamfile, femfile, outfile):
    fm  = meshio.read(foamfile)
    fp  = fm.points
    ft  = fm.points[fm.cells_dict["triangle"]] if "triangle" not in fm.cells_dict else fp
    ft  = fm.point_data["traction"]
    fp  = fm.points

    sm  = meshio.read(femfile)
    sp  = sm.points * 1e-3
    st  = sm.cells_dict["triangle"]

    interp         = RBFInterpolator(fp, ft, kernel="thin_plate_spline", neighbors=20)
    nodal_traction = interp(sp)

    trib_area = np.zeros(len(sp))
    for tri in st:
        P0, P1, P2 = sp[tri[0]], sp[tri[1]], sp[tri[2]]
        area = 0.5 * np.linalg.norm(np.cross(P1 - P0, P2 - P0))
        for j in tri:
            trib_area[j] += area / 3.0
    nodal_force = nodal_traction * trib_area[:, np.newaxis]

    foam_tri  = fm.cells_dict["triangle"]
    foam_area = 0.5 * np.linalg.norm(
        np.cross(fp[foam_tri[:, 1]] - fp[foam_tri[:, 0]],
                 fp[foam_tri[:, 2]] - fp[foam_tri[:, 0]]), axis=1)
    foam_tract = ft[foam_tri].mean(axis=1)
    foam_force = (foam_tract * foam_area[:, np.newaxis]).sum(axis=0)
    fem_force  = nodal_force.sum(axis=0)
    err        = np.linalg.norm(fem_force - foam_force) / np.linalg.norm(foam_force)
    print(f"[MAP] FOAM force [N] : {foam_force}")
    print(f"[MAP] FEM  force [N] : {fem_force}")
    print(f"[MAP] Force error    : {err*100:.2f}%")

    meshio.write(outfile, meshio.Mesh(
        points=sp,
        cells=[("triangle", st)],
        point_data={"traction": nodal_traction},
    ))
    print(f"[MAP] Saved =====> {outfile}")
    return nodal_force


def load_traction_xdmf(xdmffile, domain, gdim):
    data   = meshio.read(xdmffile)
    pts    = data.points
    tract  = data.point_data["traction"]

    VT     = fem.functionspace(domain, ("Lagrange", 1, (gdim,)))
    f      = fem.Function(VT, name="traction")
    coords = VT.tabulate_dof_coordinates()

    print(f"[LOAD] FEM pts range : {domain.geometry.x.min(axis=0)} =====> {domain.geometry.x.max(axis=0)}")
    print(f"[LOAD] XDMF pts range : {pts.min(axis=0)} =====> {pts.max(axis=0)}")

    tree = cKDTree(pts)
    dist, idx = tree.query(coords, k=1)
    print(f"[LOAD] Max KDTree dist : {dist.max():.4e} m   (should be < 1e-3 m)")

    f.x.array[:] = tract[idx].flatten()
    f.x.scatter_forward()
    return f





# FAILURE CRITERIA (post-processing)
def tsai_wu(sigma_mat, strengths):
    s1, s2, s6 = sigma_mat
    Xt, Xc     = strengths["Xt"], strengths["Xc"]
    Yt, Yc     = strengths["Yt"], strengths["Yc"]
    S          = strengths["S"]

    F1  =  1/Xt - 1/Xc
    F2  =  1/Yt - 1/Yc
    F11 =  1/(Xt*Xc)
    F22 =  1/(Yt*Yc)
    F66 =  1/S**2
    F12 = -0.5 / np.sqrt(Xt * Xc * Yt * Yc)

    return (F1*s1 + F2*s2
            + F11*s1**2 + F22*s2**2
            + F66*s6**2 + 2*F12*s1*s2)


def hashin(sigma_mat, strengths):
    s1, s2, s6 = sigma_mat
    Xt, Xc     = strengths["Xt"], strengths["Xc"]
    Yt, Yc     = strengths["Yt"], strengths["Yc"]
    SL         = strengths["S"]
    ST         = strengths.get("ST", Yc / 2.0)

    out = {}
    if s1 >= 0:
        out["fiber_t"] = (s1/Xt)**2 + (s6/SL)**2
    else:
        out["fiber_c"] = (s1/Xc)**2

    if s2 >= 0:
        out["matrix_t"] = (s2/Yt)**2 + (s6/SL)**2
    else:
        out["matrix_c"] = ((s2/(2*ST))**2 + (Yc/(2*ST))**2 * (s2/Yc) + (s6/SL)**2)
    return out


def recover_and_evaluate_failure(domain, v_sol, mat, strengths, criterion="tsai_wu"):
    assert mat.kind == "clt", "Failure recovery requires CLT material"

    u_h, theta_h = ufl.split(v_sol)
    e1, e2, e3   = local_frame(domain, domain.geometry.dim)
    P            = tangent_projection(e1, e2)
    eps_h, _     = membrane_strain(u_h, P)
    kappa_h      = bending_strain(theta_h, e3, P)

    DG0    = fem.functionspace(domain, ("DG", 0, (3,)))
    eps_fn = fem.Function(DG0)
    kap_fn = fem.Function(DG0)

    eps_fn.interpolate(fem.Expression(to_voigt(eps_h),   DG0.element.interpolation_points))
    kap_fn.interpolate(fem.Expression(to_voigt(kappa_h), DG0.element.interpolation_points))

    eps0_vals  = eps_fn.x.array.reshape(-1, 3)
    kappa_vals = kap_fn.x.array.reshape(-1, 3)

    layup   = mat._layup_angles
    t_ply   = mat._t_ply
    Q_ply   = _Q_ply(mat._E1, mat._E2, mat._G12, mat._nu12)

    H   = mat.H
    z   = -H / 2.0
    FI_all = []

    for angle in layup:
        z_mid    = z + t_ply / 2.0
        strain_k = eps0_vals + z_mid * kappa_vals

        Qb_lam    = _Qbar_ply(Q_ply, angle)
        stress_lam = strain_k @ Qb_lam.T

        a = np.radians(angle)
        c, s = np.cos(a), np.sin(a)
        T = np.array([
            [ c**2,  s**2,  2*c*s       ],
            [ s**2,  c**2, -2*c*s       ],
            [-c*s,   c*s,   c**2 - s**2 ],
        ])
        stress_mat = stress_lam @ T.T

        if criterion == "tsai_wu":
            FI_k = np.array([tsai_wu(s, strengths) for s in stress_mat])
        else:
            FI_k = np.array([max(hashin(s, strengths).values()) for s in stress_mat])

        FI_all.append(FI_k)
        print(f"  ply {angle:+4.0f} deg  max FI = {FI_k.max():.4f}")
        z += t_ply

    FI_all = np.array(FI_all)
    FI_max = FI_all.max()
    print(f"\n[FAIL] Global max FI : {FI_max:.4f}  =====>  SF = {1/FI_max:.2f}")
    return FI_max, FI_all






MESHFILE = "skin.msh"
FOAMFILE = "../wing.vtp"
XDMFFILE = "FOAMData.xdmf"
MAPFILE  = "MappedTraction.xdmf"

# LOAD MESH
MESH = gmsh.read_from_msh(MESHFILE, comm=MPI.COMM_WORLD, gdim=3)
DOMAIN = MESH.mesh
DOMAIN.geometry.x[:] *= 1E-3
CELL_TAGS = MESH.cell_tags
FACET_TAGS = MESH.facet_tags
GDIM = DOMAIN.geometry.dim
TDIM = DOMAIN.topology.dim
FDIM = TDIM - 1


# LOCAL FRAME
E1, E2, E3 = local_frame(DOMAIN, GDIM)

RESULTS_FOLDER = Path("LocalFrame")
RESULTS_FOLDER .mkdir(exist_ok=True, parents=True)
with io.VTKFile(MPI.COMM_WORLD, RESULTS_FOLDER / "LocalFrame.pvd", "w") as vtk_f:
    vtk_f.write_mesh(DOMAIN)
    vtk_f.write_function(E1, 0.0)
    vtk_f.write_function(E2, 0.0)
    vtk_f.write_function(E3, 0.0)

# FUNCTION SPACE
Ue          = basix.ufl.element("P",  DOMAIN.basix_cell(), 2, shape=(GDIM,))
Te          = basix.ufl.element("CR", DOMAIN.basix_cell(), 1, shape=(GDIM,))
V           = fem.functionspace(DOMAIN, basix.ufl.mixed_element([Ue, Te]))
v           = fem.Function(V)
u, theta    = ufl.split(v)
v_          = ufl.TestFunction(V)
u_, theta_  = ufl.split(v_)
dv          = ufl.TrialFunction(V)

# SHELL KINEMATIC
eps, kappa, gamma, drilling_strain = shell_strains(u, theta, E1, E2, E3)
eps_             = ufl.derivative(eps,            v, v_)
kappa_           = ufl.derivative(kappa,          v, v_)
gamma_           = ufl.derivative(gamma,          v, v_)
drilling_strain_ = ufl.replace(drilling_strain,  {v: v_})

# MATERIAL
# Toggle between isotropic and CLT by commenting/uncommenting one block.
# OPTION A: ISOTROPIC MATERIAL
# PURPOSE: USED FOR VALIDATION
# USAGE: MAT = isotropic_material(thickness, young, poisson, domain=DOMAIN)
# OPTION B: CLT COMPOSITE
# PURPOSE: USED FOR MAIN OPTIMIZATION
_E1, _E2, _G12, _nu12 = 181E9, 10.3E9, 7.17E9, 0.28
_t_ply  = 0.75E-3
_layup  = [0, 45, -45, 90, 90, -45, 45, 0]

MAT = clt_material(
    layup_angles = _layup,
    t_ply        = _t_ply,
    E1  = _E1,  E2  = _E2,  G12 = _G12,  nu12 = _nu12,
    G13 = _G12, G23 = _G12 * 0.5,
    kappa_s = 5/6,
)

MAT._layup_angles = _layup
MAT._t_ply  = _t_ply
MAT._E1     = _E1;  MAT._E2  = _E2
MAT._G12    = _G12; MAT._nu12 = _nu12

N, M, Q = stress_resultants(MAT, eps, kappa, gamma)
_, drilling_stress = drilling_terms(MAT, DOMAIN, drilling_strain)

ROOT_FACETS = FACET_TAGS.find(11)

Vu, _         = V.sub(0).collapse()
root_dofs_u   = fem.locate_dofs_topological((V.sub(0), Vu), FDIM, ROOT_FACETS)
uD            = fem.Function(Vu);  uD.x.array[:] = 0.0

Vt, _         = V.sub(1).collapse()
root_dofs_t   = fem.locate_dofs_topological((V.sub(1), Vt), FDIM, ROOT_FACETS)
thetaD        = fem.Function(Vt);  thetaD.x.array[:] = 0.0

BCS = [
    fem.dirichletbc(uD,     root_dofs_u, V.sub(0)),
    fem.dirichletbc(thetaD, root_dofs_t, V.sub(1)),
]


# LOAD AND MAP TRACTION FROM OPENFOAM
import_foam_traction(FOAMFILE, XDMFFILE, verbose=True)
map_traction(XDMFFILE, MESHFILE, MAPFILE)
FTraction = load_traction_xdmf(MAPFILE, DOMAIN, GDIM)




# WEAK FORMULATION
dx = ufl.Measure("dx", DOMAIN)

a_int = (
    ufl.inner(N, eps_)
    + ufl.inner(M, kappa_)
    + ufl.inner(Q, gamma_)
    + drilling_stress * drilling_strain_
) * dx

L_ext    = ufl.dot(FTraction, u_) * dx
residual = a_int - L_ext
tangent  = ufl.derivative(residual, v, dv)



# SOLVE
problem = NonlinearProblem(
    F=residual, u=v, bcs=BCS, J=tangent,
    petsc_options_prefix="wing",
    petsc_options={
        "ksp_type"                   : "preonly",
        "pc_type"                    : "lu",
        "pc_factor_mat_solver_type"  : "mumps",
        "snes_type"                  : "newtonls",
        "snes_rtol"                  : 1e-8,
        "snes_atol"                  : 1e-8,
        "snes_max_it"                : 25,
        "snes_monitor"               : None,
    },
)
problem.solve()

converged = problem.solver.getConvergedReason()
n_iter    = problem.solver.getIterationNumber()
print(f"[SNES] converged reason : {converged}")
print(f"[SNES] iterations       : {n_iter}")
assert converged > 0, f"Solver did not converge (reason {converged})"

# POST-PROCESSING
disp = v.sub(0).collapse()
rota = v.sub(1).collapse()
disp.name = "Displacement"
rota.name = "Rotation"

vdim_u   = disp.function_space.element.value_shape[0]
disp_arr = disp.x.array.reshape(-1, vdim_u)
disp_mag = np.linalg.norm(disp_arr, axis=1)
local_max  = disp_mag.max()
global_max = MPI.COMM_WORLD.allreduce(local_max, op=MPI.MAX)
print(f"\n[POST] Max displacement magnitude : {global_max:.6e} m")

if MAT.kind == "clt":
    STRENGTHS = {
        "Xt": 1500e6, "Xc": 900e6,
        "Yt":   50e6, "Yc": 200e6,
        "S":    70e6,
    }
    print("\n[POST] Tsai-Wu failure indices per ply:")
    FI_max, FI_all = recover_and_evaluate_failure(
        DOMAIN, v, MAT, STRENGTHS, criterion="tsai_wu"
    )

# EXPORT SOLUTION 
RESULT_FOLDER = Path("Result")
RESULT_FOLDER.mkdir(exist_ok=True, parents=True)

Vout      = fem.functionspace(DOMAIN, ("Lagrange", 1, (GDIM,)))
disp_out  = fem.Function(Vout); disp_out.interpolate(disp); disp_out.name = "Displacement"
rota_out  = fem.Function(Vout); rota_out.interpolate(rota);  rota_out.name = "Rotation"

with io.XDMFFile(MPI.COMM_WORLD, RESULT_FOLDER / "results.xdmf", "w") as xdmf:
    xdmf.write_mesh(DOMAIN)
    xdmf.write_function(disp_out)
    xdmf.write_function(rota_out)

print(f"[EXPORT] Results written to {RESULT_FOLDER / 'results.xdmf'}")

Info    : Reading 'skin.msh'...
Info    : 3280 nodes
Info    : 6560 elements
Info    : Done reading 'skin.msh'
[CLT] Layup  : [0, 45, -45, 90, 90, -45, 45, 0]
[CLT] H      : 6.00 mm
[CLT] max|B| : 9.82e-11  SYMMETRIC
[CLT] A11    : 458.21 MPa·m
[CLT] D11    : 2309.1921 N·m²
[FOAM] Total OpenFOAM force: [ -1159.2976   4449.0146 -17788.746 ]
[FOAM] Exported traction to the file FOAMData.xdmf

[MAP] FOAM force [N] : [ -1159.4213   4448.869  -17789.004 ]
[MAP] FEM  force [N] : [ -1200.00438549   4606.87837857 -17879.90229553]
[MAP] Force error    : 1.02%
[MAP] Saved =====> MappedTraction.xdmf
[LOAD] FEM pts range : [ 1.13686838e-16  2.27373675e-16 -7.78196237e-02] =====> [2.499403   3.         0.04446707]
[LOAD] XDMF pts range : [ 1.13686838e-16  2.27373675e-16 -7.78196237e-02] =====> [2.499403   3.         0.04446707]
[LOAD] Max KDTree dist : 6.2808e-16 m   (should be < 1e-3 m)
  0 SNES Function norm 2.344337734797e+03
  1 SNES Function norm 3.934354071440e-06
[SNES] converged reason : 3


In [3]:
# Export membrane strains and curvatures
u_h, theta_h = ufl.split(v)
e1, e2, e3 = E1, E2, E3
P = tangent_projection(e1, e2)

eps_h, _ = membrane_strain(u_h, P)
kap_h    = bending_strain(theta_h, e3, P)

DG0_3 = fem.functionspace(DOMAIN, ("DG", 0, (3,)))

epsV = fem.Function(DG0_3, name="eps_voigt")
kapV = fem.Function(DG0_3, name="kappa_voigt")

epsV.interpolate(fem.Expression(to_voigt(eps_h), DG0_3.element.interpolation_points))
kapV.interpolate(fem.Expression(to_voigt(kap_h), DG0_3.element.interpolation_points))
epsV.x.scatter_forward()
kapV.x.scatter_forward()

N_h, M_h, Q_h = stress_resultants(MAT, eps_h, kap_h, shear_strain(u_h, theta_h, e3, P))

NV = fem.Function(DG0_3, name="N_voigt")
MV = fem.Function(DG0_3, name="M_voigt")

NV.interpolate(fem.Expression(to_voigt(N_h), DG0_3.element.interpolation_points))
MV.interpolate(fem.Expression(to_voigt(M_h), DG0_3.element.interpolation_points))
NV.x.scatter_forward()
MV.x.scatter_forward()

DIAGNOSTICS_FOLDER = Path("Diagnostics")
with io.XDMFFile(MPI.COMM_WORLD, DIAGNOSTICS_FOLDER / "StressField.xdmf", "w") as xdmf:
    xdmf.write_mesh(DOMAIN)
    xdmf.write_function(epsV)
    xdmf.write_function(kapV)
    xdmf.write_function(NV)
    xdmf.write_function(MV)

In [5]:
# Reaction forces at the clamped root (robust method)

from petsc4py import PETSc
F_int = fem.petsc.assemble_vector(fem.form(a_int))
F_int.ghostUpdate(addv=PETSc.InsertMode.ADD,
                  mode=PETSc.ScatterMode.REVERSE)

F_ext = fem.petsc.assemble_vector(fem.form(L_ext))
F_ext.ghostUpdate(addv=PETSc.InsertMode.ADD,
                  mode=PETSc.ScatterMode.REVERSE)

R = F_int.copy()
R.axpy(-1.0, F_ext)

root_u_global = fem.locate_dofs_topological(V.sub(0), FDIM, ROOT_FACETS)

R_array = R.array
Ru = R_array[root_u_global]
Ru = Ru.reshape(-1, GDIM)

local_reaction = Ru.sum(axis=0)
global_reaction = MPI.COMM_WORLD.allreduce(local_reaction, op=MPI.SUM)

if MPI.COMM_WORLD.rank == 0:
    print("[REACTION] Root reaction force [N] =", global_reaction)

[REACTION] Root reaction force [N] = [ 1200.00438653 -4606.87837583 17879.90228539]


In [7]:
# Reaction moments (optional but recommended)
x = ufl.SpatialCoordinate(DOMAIN)
x0 = ufl.as_vector([0.4, 0.0, 0.0])  # reference point

r = x - x0
m = ufl.cross(r, FTraction)  # vector-valued

Mext = np.zeros(3)

for i in range(3):
    Mi = fem.assemble_scalar(fem.form(m[i] * dx))
    Mext[i] = Mi

Mext = MPI.COMM_WORLD.allreduce(Mext, op=MPI.SUM)

if MPI.COMM_WORLD.rank == 0:
    print("[CHECK] Applied moment [N*m] =", Mext)

[CHECK] Applied moment [N*m] = [-27887.20966821  17914.89004059    996.45958883]


In [8]:
# ============================
# POST / VALIDATION DIAGNOSTICS
# Drop this block AFTER problem.solve() and AFTER you have:
#   - DOMAIN, GDIM, FDIM, ROOT_FACETS, V, v, u_, theta_, dx
#   - E1,E2,E3 (local frame Functions)
#   - MAT (clt material)
#   - N,M,Q, eps,kappa,gamma (or can rebuild)
#   - a_int, L_ext already defined (as in your file)
#   - FTraction already loaded
# ============================

from petsc4py import PETSc
from dolfinx import fem, io
import ufl
import numpy as np
from pathlib import Path
from mpi4py import MPI

comm = MPI.COMM_WORLD

def _assemble_vec(form):
    vec = fem.petsc.assemble_vector(fem.form(form))
    vec.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    return vec

def _assemble_scalar(expr):
    val = fem.assemble_scalar(fem.form(expr))
    return comm.allreduce(val, op=MPI.SUM)

def _max_global(arr_local):
    return comm.allreduce(float(np.max(arr_local)), op=MPI.MAX)

def _min_global(arr_local):
    return comm.allreduce(float(np.min(arr_local)), op=MPI.MIN)

# -------- 1) Rebuild kinematics at converged solution (robust)
u_h, theta_h = ufl.split(v)

# Tangent basis
e1, e2, e3 = E1, E2, E3
def hstack(vecs):
    return ufl.as_matrix([[vi[i] for i in range(len(vi))] for vi in vecs]).T
def tangent_projection(e1, e2):
    return hstack([e1, e2])
def tangential_gradient(w, P):
    return ufl.dot(ufl.grad(w), P)
def membrane_strain(u, P):
    t_gu = ufl.dot(P.T, tangential_gradient(u, P))
    return ufl.sym(t_gu), t_gu
def bending_strain(theta, e3, P):
    beta = ufl.cross(e3, theta)
    return ufl.sym(ufl.dot(P.T, tangential_gradient(beta, P)))
def shear_strain(u, theta, e3, P):
    beta = ufl.cross(e3, theta)
    return tangential_gradient(ufl.dot(u, e3), P) - ufl.dot(P.T, beta)

def to_voigt_2x2(e):
    return ufl.as_vector([e[0,0], e[1,1], 2.0*e[0,1]])

P = tangent_projection(e1, e2)
eps_h, _ = membrane_strain(u_h, P)
kap_h    = bending_strain(theta_h, e3, P)
gam_h    = shear_strain(u_h, theta_h, e3, P)

# -------- 2) Export membrane strain / curvature / resultants (DG0 Voigt)
DG0_3 = fem.functionspace(DOMAIN, ("DG", 0, (3,)))

epsV = fem.Function(DG0_3, name="eps_voigt")     # [eps11, eps22, 2eps12]
kapV = fem.Function(DG0_3, name="kappa_voigt")   # [k11, k22, 2k12]

epsV.interpolate(fem.Expression(to_voigt_2x2(eps_h), DG0_3.element.interpolation_points))
kapV.interpolate(fem.Expression(to_voigt_2x2(kap_h), DG0_3.element.interpolation_points))
epsV.x.scatter_forward()
kapV.x.scatter_forward()

# Recompute N,M,Q at the converged fields (use your stress_resultants if in scope)
def from_voigt_2x2(vv):
    return ufl.as_tensor([[vv[0], vv[2]],[vv[2], vv[1]]])

def stress_resultants_clt(mat, eps, kappa, gamma):
    eps_v   = to_voigt_2x2(eps)
    kappa_v = to_voigt_2x2(kappa)
    # mat.A_ufl etc are 3x3 / As_ufl is 2x2
    N_v = ufl.dot(mat.A_ufl, eps_v) + ufl.dot(mat.B_ufl, kappa_v)
    M_v = ufl.dot(mat.B_ufl, eps_v) + ufl.dot(mat.D_ufl, kappa_v)
    N   = from_voigt_2x2(N_v)
    M   = from_voigt_2x2(M_v)
    Q   = ufl.dot(mat.As_ufl, gamma)  # 2-vector
    return N, M, Q

N_h, M_h, Q_h = stress_resultants_clt(MAT, eps_h, kap_h, gam_h)

NV = fem.Function(DG0_3, name="N_voigt")   # [N11, N22, 2N12]
MV = fem.Function(DG0_3, name="M_voigt")   # [M11, M22, 2M12]
NV.interpolate(fem.Expression(to_voigt_2x2(N_h), DG0_3.element.interpolation_points))
MV.interpolate(fem.Expression(to_voigt_2x2(M_h), DG0_3.element.interpolation_points))
NV.x.scatter_forward()
MV.x.scatter_forward()

# -------- 3) Quantitative ranges (global min/max) for eps components
eps_vals = epsV.x.array.reshape(-1, 3)
kap_vals = kapV.x.array.reshape(-1, 3)

eps_min = np.array([_min_global(eps_vals[:,i]) for i in range(3)])
eps_max = np.array([_max_global(eps_vals[:,i]) for i in range(3)])
kap_min = np.array([_min_global(kap_vals[:,i]) for i in range(3)])
kap_max = np.array([_max_global(kap_vals[:,i]) for i in range(3)])

if comm.rank == 0:
    print("\n[DIAG] Membrane strain eps_voigt ranges:")
    print(f"  eps11: [{eps_min[0]:+.3e}, {eps_max[0]:+.3e}]")
    print(f"  eps22: [{eps_min[1]:+.3e}, {eps_max[1]:+.3e}]")
    print(f"  2eps12:[{eps_min[2]:+.3e}, {eps_max[2]:+.3e}]  (=> eps12 half)")
    print("\n[DIAG] Curvature kappa_voigt ranges [1/m]:")
    print(f"  k11:   [{kap_min[0]:+.3e}, {kap_max[0]:+.3e}]")
    print(f"  k22:   [{kap_min[1]:+.3e}, {kap_max[1]:+.3e}]")
    print(f"  2k12:  [{kap_min[2]:+.3e}, {kap_max[2]:+.3e}]  (=> k12 half)")

# -------- 4) Compare max strain vs simple material strain limits (very rough)
# Xt/E1, Yt/E2, S/G12 (engineering strain scale indicators)
Xt = 1500e6; Yt = 50e6; S12 = 70e6
E1v = float(MAT._E1); E2v = float(MAT._E2); G12v = float(MAT._G12)

eps1_lim = Xt / E1v
eps2_lim = Yt / E2v
gam_lim  = S12 / G12v  # engineering shear strain gamma12 approx

eps11_absmax = _max_global(np.abs(eps_vals[:,0]))
eps22_absmax = _max_global(np.abs(eps_vals[:,1]))
eps12_absmax = _max_global(np.abs(eps_vals[:,2]) * 0.5)  # because stored 2eps12

if comm.rank == 0:
    print("\n[DIAG] Simple strain-limit sanity (NOT a failure criterion):")
    print(f"  |eps11|max = {eps11_absmax:.3e}  vs Xt/E1 ≈ {eps1_lim:.3e}")
    print(f"  |eps22|max = {eps22_absmax:.3e}  vs Yt/E2 ≈ {eps2_lim:.3e}")
    print(f"  |eps12|max = {eps12_absmax:.3e}  vs S/G12 ≈ {gam_lim:.3e}")

# -------- 5) Energy partition (membrane vs bending vs shear)
# Use the *resultants* N,M,Q and the strains eps,kappa,gamma
U_mem = 0.5 * _assemble_scalar(ufl.inner(N_h, eps_h) * dx)
U_ben = 0.5 * _assemble_scalar(ufl.inner(M_h, kap_h) * dx)
U_shr = 0.5 * _assemble_scalar(ufl.dot(Q_h, gam_h) * dx)
U_tot = U_mem + U_ben + U_shr

# External work: W = 0.5 ∫ t·u
W_ext = 0.5 * _assemble_scalar(ufl.dot(FTraction, u_h) * dx)

if comm.rank == 0:
    print("\n[DIAG] Energy partition:")
    print(f"  U_mem = {U_mem:.6e} J")
    print(f"  U_ben = {U_ben:.6e} J")
    print(f"  U_shr = {U_shr:.6e} J")
    print(f"  U_tot = {U_tot:.6e} J")
    print(f"  W_ext = {W_ext:.6e} J  (linear check: U_tot ≈ W_ext)")

# -------- 6) Global force + moment from traction (about chosen reference point)
# Set x0 to match OpenFOAM 'origin' (units must match your FEM coordinates)
x = ufl.SpatialCoordinate(DOMAIN)
x0 = ufl.as_vector([0.4, 0.0, 0.0])  # <-- match OpenFOAM origin
r  = x - x0

F_app = np.zeros(3)
M_app = np.zeros(3)
for i in range(3):
    F_app[i] = _assemble_scalar(FTraction[i] * dx)
m = ufl.cross(r, FTraction)
for i in range(3):
    M_app[i] = _assemble_scalar(m[i] * dx)

if comm.rank == 0:
    print("\n[CHECK] Applied force from traction [N] =", F_app)
    print("[CHECK] Applied moment about x0 [N*m] =", M_app)

# -------- 7) Reaction forces at clamp (from F_int - F_ext, DOFs on Dirichlet boundary)
# Build consistent internal/external vectors using your already-defined a_int and L_ext
F_int_vec = _assemble_vec(a_int)
F_ext_vec = _assemble_vec(L_ext)
R = F_int_vec.copy()
R.axpy(-1.0, F_ext_vec)  # R = F_int - F_ext

# Displacement DOFs at root => reaction forces
root_u = fem.locate_dofs_topological(V.sub(0), FDIM, ROOT_FACETS)
Ru = R.array[root_u].reshape(-1, GDIM)
F_react = comm.allreduce(Ru.sum(axis=0), op=MPI.SUM)

# Rotation DOFs at root => reaction moments (interpretation depends on formulation)
root_t = fem.locate_dofs_topological(V.sub(1), FDIM, ROOT_FACETS)
Rt = R.array[root_t].reshape(-1, GDIM)
M_react_like = comm.allreduce(Rt.sum(axis=0), op=MPI.SUM)

if comm.rank == 0:
    print("\n[REACTION] Root reaction force [N] =", F_react)
    print("[REACTION] Root rotation-DOF reaction (moment-like) =", M_react_like)
    print("[CHECK] Force closure (F_react + F_app) =", F_react + F_app)

# -------- 8) Export diagnostics to XDMF for visualization
DIAG_FOLDER = Path("Result")
DIAG_FOLDER.mkdir(exist_ok=True, parents=True)

with io.XDMFFile(comm, DIAG_FOLDER / "diagnostics.xdmf", "w") as xdmf:
    xdmf.write_mesh(DOMAIN)
    xdmf.write_function(epsV)
    xdmf.write_function(kapV)
    xdmf.write_function(NV)
    xdmf.write_function(MV)

if comm.rank == 0:
    print(f"\n[EXPORT] Diagnostics written to {DIAG_FOLDER / 'diagnostics.xdmf'}")


[DIAG] Membrane strain eps_voigt ranges:
  eps11: [-5.041e-04, +5.922e-04]
  eps22: [-1.915e-03, +1.276e-03]
  2eps12:[-1.396e-03, +2.619e-03]  (=> eps12 half)

[DIAG] Curvature kappa_voigt ranges [1/m]:
  k11:   [-3.977e-01, +8.675e-01]
  k22:   [-2.276e-01, +7.698e-01]
  2k12:  [-1.184e+00, +6.947e-01]  (=> k12 half)

[DIAG] Simple strain-limit sanity (NOT a failure criterion):
  |eps11|max = 5.922e-04  vs Xt/E1 ≈ 8.287e-03
  |eps22|max = 1.915e-03  vs Yt/E2 ≈ 4.854e-03
  |eps12|max = 1.309e-03  vs S/G12 ≈ 9.763e-03

[DIAG] Energy partition:
  U_mem = 4.719224e+02 J
  U_ben = 4.692095e+02 J
  U_shr = 1.017404e+01 J
  U_tot = 9.513059e+02 J
  W_ext = 9.538701e+02 J  (linear check: U_tot ≈ W_ext)

[CHECK] Applied force from traction [N] = [ -1200.00438549   4606.87837857 -17879.90229553]
[CHECK] Applied moment about x0 [N*m] = [-27887.20966821  17914.89004059    996.45958883]

[REACTION] Root reaction force [N] = [ 1200.00438653 -4606.87837583 17879.90228539]
[REACTION] Root rotation-

In [9]:
# reports whether epsilon_11 changes sign across the laminate thickness,
def ply_through_thickness_check(domain, v_sol, mat, strengths, out_xdmf="Result/ply_through_thickness.xdmf"):
    """
    Exports per-ply top/bottom eps11 and Tsai-Wu FI fields (DG0) and prints sign-change diagnostics.
    Assumes CLT material and your existing _Q_ply, _Qbar_ply, tsai_wu, local_frame, membrane_strain, bending_strain.
    """
    assert mat.kind == "clt", "Requires CLT material"

    comm = domain.comm
    gdim = domain.geometry.dim

    # --- kinematics at converged solution
    u_h, theta_h = ufl.split(v_sol)
    e1, e2, e3   = local_frame(domain, gdim)
    P            = tangent_projection(e1, e2)

    eps_h, _ = membrane_strain(u_h, P)          # 2x2
    kap_h    = bending_strain(theta_h, e3, P)   # 2x2

    # Put eps0 and kappa into DG0 arrays (cellwise constant, robust)
    DG0_3 = fem.functionspace(domain, ("DG", 0, (3,)))
    eps0_fn = fem.Function(DG0_3, name="eps0_voigt")
    kap_fn  = fem.Function(DG0_3, name="kappa_voigt")
    eps0_fn.interpolate(fem.Expression(to_voigt(eps_h), DG0_3.element.interpolation_points))
    kap_fn.interpolate(fem.Expression(to_voigt(kap_h), DG0_3.element.interpolation_points))
    eps0_fn.x.scatter_forward()
    kap_fn.x.scatter_forward()

    eps0_vals = eps0_fn.x.array.reshape(-1, 3)   # [e11,e22,2e12]
    kap_vals  = kap_fn.x.array.reshape(-1, 3)    # [k11,k22,2k12]

    # --- laminate data
    layup = mat._layup_angles
    t_ply = mat._t_ply
    Qply  = _Q_ply(mat._E1, mat._E2, mat._G12, mat._nu12)

    H = mat.H
    z_bot = -H/2.0

    # Output fields (DG0 scalar) per ply / surface
    DG0 = fem.functionspace(domain, ("DG", 0))
    fields = []

    # We'll also track global extrema to decide sign-change across thickness
    # (use eps11 only; you can extend similarly for sigma1 if you want)
    global_eps11_min_overall = +1e300
    global_eps11_max_overall = -1e300

    # Create XDMF file
    out_path = Path(out_xdmf)
    out_path.parent.mkdir(exist_ok=True, parents=True)

    # Helper to create DG0 function from numpy cell values
    def make_DG0(name, cell_vals):
        f = fem.Function(DG0, name=name)
        f.x.array[:] = cell_vals
        f.x.scatter_forward()
        return f

    # --- loop plies
    for k, angle in enumerate(layup, start=1):
        z0 = z_bot
        z1 = z_bot + t_ply

        # top/bottom z-locations of THIS ply
        z_top = z1
        z_bot_p = z0

        # strains in laminate axes at top/bottom: eps(z)=eps0 + z*kappa
        # Voigt: [e11,e22,2e12]
        eps_top = eps0_vals + z_top   * kap_vals
        eps_bot = eps0_vals + z_bot_p * kap_vals

        # stresses in laminate axes using Qbar(angle): sigma = Qbar * eps
        Qbar = _Qbar_ply(Qply, angle)      # 3x3
        sig_top_lam = eps_top @ Qbar.T
        sig_bot_lam = eps_bot @ Qbar.T

        # transform stresses to material axes (1-2) for failure:
        a = np.radians(angle)
        c, s = np.cos(a), np.sin(a)
        T = np.array([
            [ c**2,  s**2,  2*c*s       ],
            [ s**2,  c**2, -2*c*s       ],
            [-c*s,   c*s,   c**2 - s**2 ],
        ])
        sig_top_mat = sig_top_lam @ T.T
        sig_bot_mat = sig_bot_lam @ T.T

        # Tsai-Wu FI at top/bottom
        FI_top = np.array([tsai_wu(s, strengths) for s in sig_top_mat])
        FI_bot = np.array([tsai_wu(s, strengths) for s in sig_bot_mat])

        # eps11 (laminate axes) top/bottom
        eps11_top = eps_top[:, 0]
        eps11_bot = eps_bot[:, 0]

        # Track global extrema for sign-change detection
        global_eps11_min_overall = min(global_eps11_min_overall, float(eps11_top.min()), float(eps11_bot.min()))
        global_eps11_max_overall = max(global_eps11_max_overall, float(eps11_top.max()), float(eps11_bot.max()))

        # Store fields
        fields.append(make_DG0(f"ply{k:02d}_ang{angle:+g}_eps11_top", eps11_top))
        fields.append(make_DG0(f"ply{k:02d}_ang{angle:+g}_eps11_bot", eps11_bot))
        fields.append(make_DG0(f"ply{k:02d}_ang{angle:+g}_FI_top",    FI_top))
        fields.append(make_DG0(f"ply{k:02d}_ang{angle:+g}_FI_bot",    FI_bot))

        # Optional: sigma1 in material axes (useful to interpret FI)
        fields.append(make_DG0(f"ply{k:02d}_ang{angle:+g}_sigma1_top", sig_top_mat[:, 0]))
        fields.append(make_DG0(f"ply{k:02d}_ang{angle:+g}_sigma1_bot", sig_bot_mat[:, 0]))

        # Print quick per-ply maxima
        if comm.rank == 0:
            print(f"[PLY {k:02d} {angle:+g}°] max FI top/bot = {FI_top.max():.4f} / {FI_bot.max():.4f} | "
                  f"eps11 top/bot max = {eps11_top.max():+.3e} / {eps11_bot.max():+.3e}")

        z_bot = z1

    # Decide if eps11 changes sign somewhere through thickness (global criterion)
    sign_change = (global_eps11_min_overall < 0.0) and (global_eps11_max_overall > 0.0)
    if comm.rank == 0:
        print("\n[THICKNESS CHECK] eps11 over all plies/surfaces:")
        print(f"  global min eps11 = {global_eps11_min_overall:+.3e}")
        print(f"  global max eps11 = {global_eps11_max_overall:+.3e}")
        print(f"  sign change across thickness?  {'YES' if sign_change else 'NO'}")

    # Export to XDMF
    with io.XDMFFile(comm, out_path, "w") as xdmf:
        xdmf.write_mesh(domain)
        for f in fields:
            xdmf.write_function(f)

    if comm.rank == 0:
        print(f"[EXPORT] Per-ply top/bottom eps11 + FI written to {out_path}")

if MAT.kind == "clt":
    STRENGTHS = {
        "Xt": 1500e6, "Xc": 900e6,
        "Yt":   50e6, "Yc": 200e6,
        "S":    70e6,
    }

    ply_through_thickness_check(
        DOMAIN, v, MAT, STRENGTHS,
        out_xdmf="Result/ply_through_thickness.xdmf"
    )

[PLY 01 +0°] max FI top/bot = 0.2602 / 0.2671 | eps11 top/bot max = +9.621e-04 / +1.255e-03
[PLY 02 +45°] max FI top/bot = 0.2263 / 0.2290 | eps11 top/bot max = +6.857e-04 / +9.621e-04
[PLY 03 -45°] max FI top/bot = 0.3201 / 0.3467 | eps11 top/bot max = +6.355e-04 / +6.857e-04
[PLY 04 +90°] max FI top/bot = 0.2557 / 0.2459 | eps11 top/bot max = +5.922e-04 / +6.355e-04
[PLY 05 +90°] max FI top/bot = 0.2658 / 0.2557 | eps11 top/bot max = +8.666e-04 / +5.922e-04
[PLY 06 -45°] max FI top/bot = 0.3152 / 0.2903 | eps11 top/bot max = +1.515e-03 / +8.666e-04
[PLY 07 +45°] max FI top/bot = 0.5315 / 0.3556 | eps11 top/bot max = +2.164e-03 / +1.515e-03
[PLY 08 +0°] max FI top/bot = 0.5396 / 0.4201 | eps11 top/bot max = +2.815e-03 / +2.164e-03

[THICKNESS CHECK] eps11 over all plies/surfaces:
  global min eps11 = -2.391e-03
  global max eps11 = +2.815e-03
  sign change across thickness?  YES
[EXPORT] Per-ply top/bottom eps11 + FI written to Result/ply_through_thickness.xdmf


In [13]:
def solve_case(alpha_value: float, t_ply_value: float):

    # --- rebuild material
    MAT_local = clt_material(
        layup_angles=_layup,
        t_ply=t_ply_value,
        E1=_E1, E2=_E2, G12=_G12, nu12=_nu12,
        G13=_G12, G23=_G12 * 0.5,
        kappa_s=5/6,
    )

    # --- local frame
    E1_loc, E2_loc, E3_loc = local_frame(DOMAIN, GDIM)

    # --- function space
    Ue = basix.ufl.element("P", DOMAIN.basix_cell(), 2, shape=(GDIM,))
    Te = basix.ufl.element("CR", DOMAIN.basix_cell(), 1, shape=(GDIM,))
    Vloc = fem.functionspace(DOMAIN, basix.ufl.mixed_element([Ue, Te]))

    vloc = fem.Function(Vloc)
    u, theta = ufl.split(vloc)
    v_test = ufl.TestFunction(Vloc)
    u_, theta_ = ufl.split(v_test)
    dv = ufl.TrialFunction(Vloc)

    # --- kinematics
    eps, kappa, gamma, drilling_strain = shell_strains(u, theta, E1_loc, E2_loc, E3_loc)
    eps_   = ufl.derivative(eps, vloc, v_test)
    kappa_ = ufl.derivative(kappa, vloc, v_test)
    gamma_ = ufl.derivative(gamma, vloc, v_test)
    drilling_strain_ = ufl.replace(drilling_strain, {vloc: v_test})

    # --- stress resultants
    N, M, Q = stress_resultants(MAT_local, eps, kappa, gamma)
    _, drilling_stress = drilling_terms(MAT_local, DOMAIN, drilling_strain)

    dx = ufl.Measure("dx", DOMAIN)

    a_int = (
        ufl.inner(N, eps_)
        + ufl.inner(M, kappa_)
        + ufl.inner(Q, gamma_)
        + drilling_stress * drilling_strain_
    ) * dx

    alpha = fem.Constant(DOMAIN, float(alpha_value))
    L_ext = alpha * ufl.dot(FTraction, u_) * dx

    residual = a_int - L_ext
    tangent = ufl.derivative(residual, vloc, dv)

    problem = NonlinearProblem(
        F=residual,
        u=vloc,
        bcs=BCS,
        J=tangent,
        petsc_options_prefix="scale",
        petsc_options={
            "ksp_type": "preonly",
            "pc_type": "lu",
            "pc_factor_mat_solver_type": "mumps",
            "snes_type": "newtonls",
        },
    )

    problem.solve()

    # --- postprocess
    disp = vloc.sub(0).collapse()
    disp_arr = disp.x.array.reshape(-1, GDIM)
    umax = MPI.COMM_WORLD.allreduce(float(np.linalg.norm(disp_arr, axis=1).max()), op=MPI.MAX)

    # strain max
    P = tangent_projection(E1_loc, E2_loc)
    eps_h, _ = membrane_strain(u, P)
    DG0_3 = fem.functionspace(DOMAIN, ("DG", 0, (3,)))
    epsV = fem.Function(DG0_3)
    epsV.interpolate(fem.Expression(to_voigt(eps_h), DG0_3.element.interpolation_points))
    epsV.x.scatter_forward()
    eps_vals = epsV.x.array.reshape(-1, 3)
    eps11max = MPI.COMM_WORLD.allreduce(float(np.max(np.abs(eps_vals[:, 0]))), op=MPI.MAX)

    # energies
    kap_h = bending_strain(theta, E3_loc, P)
    gam_h = shear_strain(u, theta, E3_loc, P)
    N_h, M_h, Q_h = stress_resultants(MAT_local, eps_h, kap_h, gam_h)

    U_mem = 0.5 * fem.assemble_scalar(fem.form(ufl.inner(N_h, eps_h) * dx))
    U_ben = 0.5 * fem.assemble_scalar(fem.form(ufl.inner(M_h, kap_h) * dx))
    U_shr = 0.5 * fem.assemble_scalar(fem.form(ufl.dot(Q_h, gam_h) * dx))
    U_tot = MPI.COMM_WORLD.allreduce(float(U_mem + U_ben + U_shr), op=MPI.SUM)

    W_ext = 0.5 * fem.assemble_scalar(fem.form(alpha * ufl.dot(FTraction, u) * dx))
    W_ext = MPI.COMM_WORLD.allreduce(float(W_ext), op=MPI.SUM)

    return umax, eps11max, U_tot, W_ext

# =====================================
# SCALING STUDY
# =====================================

baseline_tply = _t_ply

cases = [
    ("BASE", 1.0, 1.0),
    ("LOADx2", 2.0, 1.0),
    ("LOADx0.5", 0.5, 1.0),
    ("THICKx0.9", 1.0, 0.9),
    ("THICKx1.1", 1.0, 1.1),
]

results = {}

print("\n=== SCALING STUDY ===")

for name, load_scale, t_scale in cases:

    umax, epsmax, U, W = solve_case(
        alpha_value=load_scale,
        t_ply_value=baseline_tply * t_scale
    )

    results[name] = (umax, epsmax, U, W)

    print(f"\n[{name}]")
    print(f"  load scale      = {load_scale}")
    print(f"  thickness scale = {t_scale}")
    print(f"  max displacement = {umax:.6e}")
    print(f"  max |eps11|      = {epsmax:.6e}")
    print(f"  U_tot            = {U:.6e}")
    print(f"  W_ext            = {W:.6e}")

# =====================================
# RATIO CHECK
# =====================================

u0, e0, U0, _ = results["BASE"]

print("\n=== RATIO CHECK ===")

def ratio(a, b):
    return a/b if b != 0 else float("nan")

for name in ["LOADx2", "LOADx0.5"]:
    u, e, U, _ = results[name]
    print(f"\n{name}:")
    print(f"  u_ratio   = {ratio(u,u0):.6f}")
    print(f"  eps_ratio = {ratio(e,e0):.6f}")
    print(f"  U_ratio   = {ratio(U,U0):.6f}")

for name in ["THICKx0.9", "THICKx1.1"]:
    u, e, U, _ = results[name]
    print(f"\n{name}:")
    print(f"  u_ratio   = {ratio(u,u0):.6f}")
    print(f"  eps_ratio = {ratio(e,e0):.6f}")


=== SCALING STUDY ===
[CLT] Layup  : [0, 45, -45, 90, 90, -45, 45, 0]
[CLT] H      : 6.00 mm
[CLT] max|B| : 9.82e-11  SYMMETRIC
[CLT] A11    : 458.21 MPa·m
[CLT] D11    : 2309.1921 N·m²

[BASE]
  load scale      = 1.0
  thickness scale = 1.0
  max displacement = 0.000000e+00
  max |eps11|      = 0.000000e+00
  U_tot            = 0.000000e+00
  W_ext            = 0.000000e+00
[CLT] Layup  : [0, 45, -45, 90, 90, -45, 45, 0]
[CLT] H      : 6.00 mm
[CLT] max|B| : 9.82e-11  SYMMETRIC
[CLT] A11    : 458.21 MPa·m
[CLT] D11    : 2309.1921 N·m²

[LOADx2]
  load scale      = 2.0
  thickness scale = 1.0
  max displacement = 0.000000e+00
  max |eps11|      = 0.000000e+00
  U_tot            = 0.000000e+00
  W_ext            = 0.000000e+00
[CLT] Layup  : [0, 45, -45, 90, 90, -45, 45, 0]
[CLT] H      : 6.00 mm
[CLT] max|B| : 9.82e-11  SYMMETRIC
[CLT] A11    : 458.21 MPa·m
[CLT] D11    : 2309.1921 N·m²

[LOADx0.5]
  load scale      = 0.5
  thickness scale = 1.0
  max displacement = 0.000000e+00
  max

In [18]:
alpha_load = fem.Constant(DOMAIN, 1.0)
L_ext = alpha_load * ufl.dot(FTraction, u_) * dx

def solve_with_scaling(load_scale, t_ply_scale):

    global MAT, N, M, Q, drilling_stress, a_int, residual, tangent, problem

    # Update load scaling
    alpha_load.value = load_scale

    # Update thickness
    t_new = _t_ply * t_ply_scale

    MAT = clt_material(
        layup_angles=_layup,
        t_ply=t_new,
        E1=_E1, E2=_E2, G12=_G12, nu12=_nu12,
        G13=_G12, G23=_G12 * 0.5,
        kappa_s=5/6,
    )

    MAT._layup_angles = _layup
    MAT._t_ply = t_new
    MAT._E1 = _E1
    MAT._E2 = _E2
    MAT._G12 = _G12
    MAT._nu12 = _nu12

    # Recompute stress resultants (using SAME eps, kappa, gamma!)
    N, M, Q = stress_resultants(MAT, eps, kappa, gamma)
    _, drilling_stress = drilling_terms(MAT, DOMAIN, drilling_strain)

    # Rebuild internal form
    a_int = (
        ufl.inner(N, eps_)
        + ufl.inner(M, kappa_)
        + ufl.inner(Q, gamma_)
        + drilling_stress * drilling_strain_
    ) * dx

    residual = a_int - L_ext
    tangent = ufl.derivative(residual, v, dv)

    problem = NonlinearProblem(
        F=residual,
        u=v,
        bcs=BCS,
        J=tangent,
        petsc_options_prefix="scale",
        petsc_options={
            "ksp_type": "preonly",
            "pc_type": "lu",
            "pc_factor_mat_solver_type": "mumps",
            "snes_type": "newtonls",
        },
    )

    v.x.array[:] = 0.0
    v.x.scatter_forward()
    problem.solve()

    # Post-process
    disp = v.sub(0).collapse()
    disp_arr = disp.x.array.reshape(-1, GDIM)
    umax = MPI.COMM_WORLD.allreduce(np.linalg.norm(disp_arr, axis=1).max(), op=MPI.MAX)

    return umax

u0 = solve_with_scaling(1.0, 1.0)
u2 = solve_with_scaling(2.0, 1.0)
u_half = solve_with_scaling(0.5, 1.0)
u_thin = solve_with_scaling(1.0, 0.9)

print("LOADx2 ratio =", u2/u0)
print("LOADx0.5 ratio =", u_half/u0)
print("THICKx0.9 ratio =", u_thin/u0)

[CLT] Layup  : [0, 45, -45, 90, 90, -45, 45, 0]
[CLT] H      : 6.00 mm
[CLT] max|B| : 9.82e-11  SYMMETRIC
[CLT] A11    : 458.21 MPa·m
[CLT] D11    : 2309.1921 N·m²
[CLT] Layup  : [0, 45, -45, 90, 90, -45, 45, 0]
[CLT] H      : 6.00 mm
[CLT] max|B| : 9.82e-11  SYMMETRIC
[CLT] A11    : 458.21 MPa·m
[CLT] D11    : 2309.1921 N·m²
[CLT] Layup  : [0, 45, -45, 90, 90, -45, 45, 0]
[CLT] H      : 6.00 mm
[CLT] max|B| : 9.82e-11  SYMMETRIC
[CLT] A11    : 458.21 MPa·m
[CLT] D11    : 2309.1921 N·m²
[CLT] Layup  : [0, 45, -45, 90, 90, -45, 45, 0]
[CLT] H      : 5.40 mm
[CLT] max|B| : 1.16e-10  SYMMETRIC
[CLT] A11    : 412.39 MPa·m
[CLT] D11    : 1683.4010 N·m²
LOADx2 ratio = 1.9999999999999991
LOADx0.5 ratio = 0.4999999999999997
THICKx0.9 ratio = 1.1133978840705836
